# Example 013: SPS kickers — resonator vs. pole-residue fitting

This notebook repeats the Driving Y fit of Example 008 and compares three ways of fitting the same data:

1. **Resonator form** (standard IDDEFIX): 6 transverse resonators, parameters $(R_s, Q, f_r)$ found by Differential Evolution (DE).
2. **Pole-residue form, fully evolutionary**: 6 second-order sections (see below). DE finds poles *and* residues.
3. **Pole-residue form, least-squares residues**: 6 second-order sections. DE finds only the poles; for every candidate the residues are solved by linear least squares.

All three use the same data, the same SmartBounds result, the same DE settings and the same objective (sum of squared complex errors). We compare accuracy and fit time.

**Pole-residue model.** Each mode is a second-order section

$$Z_k(s) = j\,\frac{b_0 + b_1 s}{s^2 + \dfrac{\omega_r}{Q}\,s + \omega_r^2}, \qquad \omega_r = 2\pi f_r, \quad s = j\omega,$$

(`pole_parametrization="resonator"`). Its poles are a complex-conjugate pair for $Q > 0.5$ and two real poles for $Q < 0.5$, so — like the resonator — a mode can move between under- and overdamped during the fit. The poles are searched in exactly the same $(f_r, Q)$ bounds as the resonators. The numerator $(b_0, b_1)$ plays the role of the residues. A transverse resonator is the special case $b_0 = R_s\omega_r^2/Q$, $b_1 = 0$.

**Model size.** A resonator has 3 parameters $(R_s, Q, f_r)$, so 6 resonators have 18 degrees of freedom. A section has 4: $(f_r, Q, b_0, b_1)$. The pole-residue models therefore have 24 parameters (fit 2: all 24 by DE; fit 3: 12 by DE, 12 by least squares).

**Differences to Example 008.**
- Example 008 uses CMA-ES. Here all three fits use SciPy DE, so that time differences come from the model form and not from the optimizer.
- The resonator DE population is not seeded with the SmartBounds estimates, because the pole-residue fits have no equivalent seed. All three start from the same Latin-hypercube initialization.
- `compute_uncertainties=False`, so the resonator timing contains only the optimization.
- All fits run single-threaded (`workers=1`, `updating="immediate"`) for comparable timings.

In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

import iddefix
from iddefix.poleResidueFitting import fit_poles_evolutionary

# change to `ipympl` for interactive notebooks
%matplotlib inline

## Load the impedance

Same data as Example 008: frequency in GHz followed by Driving X, Driving Y, Detuning X and Detuning Y (Ω/m). We fit Driving Y.

In [ ]:
impedance_file = Path("data/008_Z_kickers_postLS2_SPSmodel.dat")
impedance_data = np.loadtxt(impedance_file, dtype=str)
frequency_ghz = impedance_data[:, 0].astype(float)
impedance_components = np.char.replace(impedance_data[:, 1:], "i", "j").astype(complex)

frequency = frequency_ghz * 1e9
driving_y_impedance = impedance_components[:, 1]

fig, axis = plt.subplots(figsize=(10, 4))
axis.plot(frequency_ghz, driving_y_impedance.real / 1e6, label="Real")
axis.plot(frequency_ghz, driving_y_impedance.imag / 1e6, label="Imaginary")
axis.set_title("Driving Y")
axis.set_xlabel("Frequency [GHz]")
axis.set_ylabel("Transverse impedance [MΩ/m]")
axis.grid(True)
axis.legend()
fig.tight_layout()
plt.show()

## SmartBounds

Identical to Example 008: modes are detected on the real part, with an extra peak added at 2 GHz. This gives 6 resonators and their $(R_s, Q, f_r)$ bounds.

In [ ]:
driving_y_bounds = iddefix.SmartBoundDetermination(
    frequency,
    np.real(driving_y_impedance),
    minimum_peak_height=1e6,
    impedance_type="real",
    plane="transverse",
)
driving_y_bounds.add_peak(2e9)

driving_y_bounds.inspect(show_bounds=False, show_components=True)
driving_y_bounds.to_table()

## Common settings

The same DE settings are used for all three fits. Note that SciPy's `popsize` is a multiplier: the actual population is `popsize × number of DE parameters`, i.e. 270 (resonators), 360 (fully evolutionary pole-residue) and 180 (least-squares pole-residue).

In [ ]:
N_MODES = driving_y_bounds.N_resonators  # 6

DE_SETTINGS = dict(
    maxiter=2000,
    popsize=15,
    mutation=(0.1, 0.5),
    crossover_rate=0.8,
    tol=0.01,
    seed=0,
)


def normalized_error(fitted, target=driving_y_impedance):
    # sqrt of the common objective: ||Z_fit - Z|| / ||Z||
    return np.linalg.norm(fitted - target) / np.linalg.norm(target)


def max_error(fitted, target=driving_y_impedance):
    # largest pointwise error, relative to the peak of |Z|
    return np.max(np.abs(fitted - target)) / np.max(np.abs(target))


results = {}

## 1. Resonator form (DE)

Same model as Example 008 (`plane="transverse"`, `objectiveFunction="complex"`, SmartBounds bounds), fitted with DE.

In [ ]:
resonator_model = iddefix.EvolutionaryAlgorithm(
    frequency,
    driving_y_impedance,
    N_resonators=N_MODES,
    parameterBounds=driving_y_bounds.parameterBounds,
    compute_uncertainties=False,
    plane="transverse",
    objectiveFunction="complex",
)

start = perf_counter()
resonator_model.run_differential_evolution(
    workers=1,
    updating="immediate",
    **DE_SETTINGS,
)
runtime = perf_counter() - start

results["Resonator (DE)"] = dict(
    impedance=resonator_model.get_impedance(),
    runtime=runtime,
    de_parameters=3 * N_MODES,
    model_parameters=3 * N_MODES,
)
print(f"runtime: {runtime:.1f} s")

## Bounds for the pole-residue form

- **Poles:** the $(f_r, Q)$ bounds of SmartBounds are used unchanged, including $Q < 0.5$.
- **Numerator (fit 2 only):** for a transverse resonator $b_0 = R_s\omega_r^2/Q$ and $b_1 = 0$. We allow both signs and bound $|b_0| \le R_{s,\max}\,\omega_{r,\max}^2/Q_{\min}$ and $|b_1| \le R_{s,\max}\,\omega_{r,\max}/Q_{\min}$ (the latter is the size $b_1$ would have in a longitudinal-type resonator).

Fit 3 needs no numerator bounds, since least squares solves for $(b_0, b_1)$ exactly.

In [ ]:
bounds = np.asarray(driving_y_bounds.parameterBounds)  # rows: Rs, Q, fres per mode
rs_max = bounds[0::3, 1]
q_min = bounds[1::3, 0]
omega_max = 2 * np.pi * bounds[2::3, 1]

# Order expected by fit_poles_evolutionary (no real poles):
# [f_r_1, Q_1, f_r_2, Q_2, ...] -- the SmartBounds bounds without Rs
pole_bounds = [tuple(bounds[3 * mode + index]) for mode in range(N_MODES) for index in (2, 1)]

# Order: [b0_1, b1_1, b0_2, b1_2, ...]
b0_limit = rs_max * omega_max**2 / q_min
b1_limit = rs_max * omega_max / q_min
residue_bounds = [
    bound
    for mode in range(N_MODES)
    for bound in ((-b0_limit[mode], b0_limit[mode]), (-b1_limit[mode], b1_limit[mode]))
]

for mode in range(N_MODES):
    (fr_low, fr_high), (q_low, q_high) = pole_bounds[2 * mode], pole_bounds[2 * mode + 1]
    print(
        f"mode {mode + 1}: f_r [{fr_low:.2e}, {fr_high:.2e}] Hz  Q [{q_low:.3f}, {q_high:.2f}]  "
        f"|b0| <= {b0_limit[mode]:.2e}  |b1| <= {b1_limit[mode]:.2e}"
    )

## 2. Pole-residue form, poles and residues by DE

All 24 parameters (12 pole, 12 numerator) are optimized by DE.

In [ ]:
start = perf_counter()
pole_residue_de = fit_poles_evolutionary(
    frequency,
    driving_y_impedance,
    number_real_poles=0,
    number_complex_pairs=N_MODES,  # number of second-order sections
    parameter_bounds=pole_bounds,
    pole_parametrization="resonator",
    plane="transverse",
    residue_solver="differential_evolution",
    residue_bounds=residue_bounds,
    **DE_SETTINGS,
)
runtime = perf_counter() - start

results["Pole-residue (DE residues)"] = dict(
    impedance=pole_residue_de.residue_fit.fitted_impedance,
    runtime=runtime,
    de_parameters=4 * N_MODES,
    model_parameters=4 * N_MODES,
)
print(f"runtime: {runtime:.1f} s, {pole_residue_de.message}")

## 3. Pole-residue form, residues by least squares

DE optimizes only the 12 pole parameters $(f_r, Q)$. For each candidate set of poles, the 12 numerator coefficients follow from a linear least-squares problem, so they are always optimal for the current poles.

In [ ]:
start = perf_counter()
pole_residue_ls = fit_poles_evolutionary(
    frequency,
    driving_y_impedance,
    number_real_poles=0,
    number_complex_pairs=N_MODES,  # number of second-order sections
    parameter_bounds=pole_bounds,
    pole_parametrization="resonator",
    plane="transverse",
    residue_solver="least_squares",
    **DE_SETTINGS,
)
runtime = perf_counter() - start

results["Pole-residue (LS residues)"] = dict(
    impedance=pole_residue_ls.residue_fit.fitted_impedance,
    runtime=runtime,
    de_parameters=2 * N_MODES,
    model_parameters=4 * N_MODES,
)
print(f"runtime: {runtime:.1f} s, {pole_residue_ls.message}")

## Comparison

- **Normalized error** $\|Z_\mathrm{fit} - Z\| / \|Z\|$ over all frequency points. This is the square root of the objective that all three fits minimize.
- **Max. error**: largest pointwise $|Z_\mathrm{fit} - Z|$ relative to $\max|Z|$. 

The DE fits are stochastic (one seed shown).

In [ ]:
header = f"{'Method':<28} {'DE params':>9} {'Model params':>12} {'Runtime [s]':>11} {'Norm. error':>12} {'Max. error':>11}"
print(header)
print("-" * len(header))
for name, result in results.items():
    de_parameters = "-" if result["de_parameters"] is None else str(result["de_parameters"])
    print(
        f"{name:<28} {de_parameters:>9} {result['model_parameters']:>12d} "
        f"{result['runtime']:>11.2f} {normalized_error(result['impedance']):>12.3e} "
        f"{max_error(result['impedance']):>11.3e}"
    )

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
styles = ["--", "-.", ":", (0, (5, 1)), (0, (1, 3))]

for axis, part, label in zip(axes[:2], (np.real, np.imag), ("Real", "Imaginary")):
    axis.plot(frequency_ghz, part(driving_y_impedance) / 1e6, "k", lw=2.5, alpha=0.4, label="Data")
    for style, (name, result) in zip(styles, results.items()):
        axis.plot(frequency_ghz, part(result["impedance"]) / 1e6, linestyle=style, label=name)
    axis.set_ylabel(f"{label} part [MΩ/m]")
    axis.grid(True)
    axis.legend()

for style, (name, result) in zip(styles, results.items()):
    error = np.abs(result["impedance"] - driving_y_impedance) / np.max(np.abs(driving_y_impedance))
    axes[2].semilogy(frequency_ghz, error, linestyle=style, label=name)
axes[2].set_ylabel(r"$|Z_\mathrm{fit} - Z| / \max|Z|$")
axes[2].set_xlabel("Frequency [GHz]")
axes[2].grid(True, which="both")
axes[2].legend()

fig.tight_layout()
plt.show()

### Fitted modes

The sections are directly comparable with the resonator table. For each section we show $(f_r, Q)$, the equivalent shunt impedance $R_s = b_0 Q/\omega_r^2$ (exact for a pure transverse resonator), and $b_1\omega_r/b_0$, which is 0 for a pure resonator and measures how much the extra numerator degree of freedom is used. Modes with $Q < 0.5$ are overdamped (two real poles).

In [ ]:
resonator_model.display_resonator_parameters()

for name, fit in (
    ("Pole-residue (DE residues)", pole_residue_de),
    ("Pole-residue (LS residues)", pole_residue_ls),
):
    print(f"\n{name}")
    print(f"{'mode':>4} {'f_r [Hz]':>11} {'Q':>8} {'R_s equiv.':>11} {'b1*w_r/b0':>10}")
    sections = fit.resonator_sections
    coefficients = fit.residue_fit.section_coefficients
    for index, ((f_r, q), (b0, b1)) in enumerate(zip(sections, coefficients), start=1):
        omega_r = 2 * np.pi * f_r
        print(f"{index:>4} {f_r:>11.3e} {q:>8.3f} {b0 * q / omega_r**2:>11.3e} {b1 * omega_r / b0:>10.3f}")